# DroneRL — Hyperparameter Analysis & Research

In [ ]:
import sys

sys.path.insert(0, '../src')
import matplotlib.pyplot as plt
import numpy as np

from dronerl.sdk import DroneRLSDK

## 1. Baseline Training (default params)

In [ ]:
sdk = DroneRLSDK('../config/setup.json')
rewards = sdk.run_headless(500)

plt.figure(figsize=(10, 4))
plt.plot(rewards)
plt.title('Reward per Episode (500 eps)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.tight_layout()
plt.show()

## 2. Sensitivity Analysis — Learning Rate (α)

In [ ]:
alphas = [0.05, 0.1, 0.3, 0.5]

plt.figure(figsize=(10, 4))
for alpha in alphas:
    sdk = DroneRLSDK('../config/setup.json')
    sdk._agent._alpha = alpha
    rewards = sdk.run_headless(300)
    plt.plot(rewards, label=f'α={alpha}')

plt.title('Sensitivity Analysis — Learning Rate (α)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.legend()
plt.tight_layout()
plt.show()

## 3. Sensitivity Analysis — Discount Factor (γ)

In [ ]:
gammas = [0.8, 0.9, 0.95, 0.99]

plt.figure(figsize=(10, 4))
for gamma in gammas:
    sdk = DroneRLSDK('../config/setup.json')
    sdk._agent._gamma = gamma
    rewards = sdk.run_headless(300)
    plt.plot(rewards, label=f'γ={gamma}')

plt.title('Sensitivity Analysis — Discount Factor (γ)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Convergence Speed Analysis

In [ ]:
sdk = DroneRLSDK('../config/setup.json')
rewards = sdk.run_headless(1000)

window = 50
threshold = 50
rolling_avg = np.convolve(rewards, np.ones(window) / window, mode='valid')

convergence_ep = next((i for i, v in enumerate(rolling_avg) if v > threshold), None)
if convergence_ep is not None:
    print(f'First episode where rolling avg > {threshold}: {convergence_ep + window}')
else:
    print(f'Rolling average never exceeded threshold of {threshold}')

plt.figure(figsize=(10, 4))
plt.plot(rewards, alpha=0.4, label='Raw reward')
plt.plot(range(window - 1, len(rewards)), rolling_avg, label=f'Rolling avg (window={window})')
plt.axhline(y=threshold, color='red', linestyle='--', label=f'Threshold = {threshold}')
if convergence_ep is not None:
    plt.axvline(x=convergence_ep + window, color='green', linestyle=':', label=f'Converged at ep {convergence_ep + window}')
plt.title('Convergence Speed Analysis (1000 eps)')
plt.xlabel('Episode')
plt.ylabel('Total Reward')
plt.legend()
plt.tight_layout()
plt.show()

## 5. Results Summary

| Experiment | Configuration | Key Observation |
|---|---|---|
| Baseline | Default params, 500 eps | Establishes reference reward curve |
| Learning Rate α | 0.05, 0.1, 0.3, 0.5 — 300 eps each | Higher α speeds up early learning but may cause instability; lower α converges more smoothly |
| Discount Factor γ | 0.8, 0.9, 0.95, 0.99 — 300 eps each | Higher γ values (0.95–0.99) prioritize long-term rewards; lower γ (0.8) focuses on immediate gains |
| Convergence | 1000 eps, rolling window=50 | First episode where rolling avg exceeds 50 indicates convergence point |